# 玩家统计数据分析

本 Notebook 用于分析 PlayerStats，输入玩家名称后展示：
- 核心指标：VPIP、PFR、WTP、AGG
- Preflop 参数分析
- Postflop 参数分析

In [1]:
# 环境初始化
import sys
from pathlib import Path

# notebook 位于 src/bayes_poker/analysis/，往上 3 层是项目根目录
project_root = Path.cwd().parent.parent.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"src 路径已添加: {src_path}")

项目根目录: /home/autumn/project/bayes_poker
src 路径已添加: /home/autumn/project/bayes_poker/src


In [2]:
# 导入依赖
import pandas as pd
from IPython.display import display, HTML

from bayes_poker.player_metrics.enums import (
    ActionType,
    Position,
    PreflopPotType,
    Street,
    TableType,
)
from bayes_poker.player_metrics.models import ActionStats, PlayerStats
from bayes_poker.player_metrics.params import PostFlopParams, PreFlopParams
from bayes_poker.player_metrics.builder import (
    calculate_aggression,
    calculate_pfr,
    calculate_total_hands,
    calculate_wtp,
)
from bayes_poker.storage.player_stats_repository import PlayerStatsRepository

print("依赖导入成功!")

依赖导入成功!


## 1. 配置与数据加载

In [3]:
# ======== 配置区域 ========

# 数据库路径（相对于项目根目录）
DB_PATH = project_root / "data" / "database" / "player_stats.db"

# 玩家名称（输入你要分析的玩家ID）
PLAYER_NAME = "scofield5656"  # <-- 修改这里

# 桌型：TableType.HEADS_UP (2人) 或 TableType.SIX_MAX (6人)
TABLE_TYPE = TableType.SIX_MAX  # <-- 修改这里

print(f"数据库路径: {DB_PATH}")
print(f"玩家名称: {PLAYER_NAME}")
print(f"桌型: {TABLE_TYPE.name}")

数据库路径: /home/autumn/project/bayes_poker/data/database/player_stats.db
玩家名称: scofield5656
桌型: SIX_MAX


In [4]:
# 加载玩家数据
def load_player_stats(db_path: Path, player_name: str, table_type: TableType) -> PlayerStats | None:
    """从数据库加载玩家统计数据。"""
    if not db_path.exists():
        print(f"❌ 数据库文件不存在: {db_path}")
        return None
    
    with PlayerStatsRepository(db_path) as repo:
        stats = repo.get(player_name, table_type)
        if stats is None:
            print(f"❌ 未找到玩家: {player_name} (桌型: {table_type.name})")
            # 列出可用玩家
            all_stats = repo.get_all(table_type)
            if all_stats:
                print(f"\n可用玩家列表 ({len(all_stats)} 人):")
                for s in sorted(all_stats, key=lambda x: x.vpip.total, reverse=True)[:20]:
                    print(f"  - {s.player_name} (手数: {s.vpip.total})")
            return None
        return stats

player_stats = load_player_stats(DB_PATH, PLAYER_NAME, TABLE_TYPE)
if player_stats:
    print(f"✅ 成功加载玩家数据: {player_stats.player_name}")

✅ 成功加载玩家数据: scofield5656


## 2. 核心指标 (VPIP / PFR / WTP / AGG)

In [5]:
def format_stat(positive: int, total: int) -> str:
    """格式化统计值为百分比字符串。"""
    if total == 0:
        return "N/A (0)"
    pct = positive / total * 100
    return f"{pct:.1f}% ({positive}/{total})"

def display_core_stats(stats: PlayerStats) -> pd.DataFrame:
    """显示核心统计指标。"""
    # 计算各项指标
    total_hands = calculate_total_hands(stats)
    vpip_pct = stats.vpip.to_float() * 100
    
    pfr_pos, pfr_total = calculate_pfr(stats)
    pfr_pct = pfr_pos / pfr_total * 100 if pfr_total > 0 else 0
    
    wtp_pos, wtp_total = calculate_wtp(stats)
    wtp_pct = wtp_pos / wtp_total * 100 if wtp_total > 0 else 0
    
    agg_pos, agg_total = calculate_aggression(stats)
    agg_pct = agg_pos / agg_total * 100 if agg_total > 0 else 0
    
    data = {
        "指标": ["总手数", "VPIP", "PFR", "WTP", "AGG"],
        "值": [
            f"{total_hands}",
            format_stat(stats.vpip.positive, stats.vpip.total),
            format_stat(pfr_pos, pfr_total),
            format_stat(wtp_pos, wtp_total),
            format_stat(agg_pos, agg_total),
        ],
        "百分比": [
            "-",
            f"{vpip_pct:.1f}%",
            f"{pfr_pct:.1f}%",
            f"{wtp_pct:.1f}%",
            f"{agg_pct:.1f}%",
        ],
        "说明": [
            "统计样本数",
            "自愿投入底池比例（Voluntarily Put money In Pot）",
            "翻前加注比例（Pre-Flop Raise）",
            "面对下注时跟注/加注比例（Went To Pot）",
            "激进度：加注数 / (加注数 + 跟注数)",
        ],
    }
    return pd.DataFrame(data)

if player_stats:
    print(f"\n📊 {player_stats.player_name} 核心统计指标\n")
    df_core = display_core_stats(player_stats)
    display(df_core.style.set_properties(**{"text-align": "left"}))


📊 scofield5656 核心统计指标



,指标,值,百分比,说明
0,总手数,2396,-,统计样本数
1,VPIP,22.2% (550/2481),22.2%,自愿投入底池比例（Voluntarily Put money In Pot）
2,PFR,27.3% (380/1392),27.3%,翻前加注比例（Pre-Flop Raise）
3,WTP,49.2% (87/177),49.2%,面对下注时跟注/加注比例（Went To Pot）
4,AGG,76.6% (193/252),76.6%,激进度：加注数 / (加注数 + 跟注数)


## 3. Preflop 参数分析

In [ ]:
def analyze_preflop_stats(stats: PlayerStats) -> pd.DataFrame:
    """分析 Preflop 统计数据。"""
    all_params = PreFlopParams.get_all_params(stats.table_type)
    
    rows = []
    for i, params in enumerate(all_params):
        action_stats = stats.preflop_stats[i]
        total = action_stats.total_samples()
        if total == 0:
            continue  # 跳过无样本的参数组合
        
        row = {
            "索引": i,
            "位置": params.position.name,
            "加注数": params.num_raises,
            "跟注数": params.num_callers,
            "前动作": params.previous_action.name,
            "IP": "是" if params.in_position_on_flop else "否",
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df.sort_values("样本数", ascending=False)

if player_stats:
    print(f"\n🃏 {player_stats.player_name} Preflop 参数分析\n")
    df_preflop = analyze_preflop_stats(player_stats)
    print(f"共 {len(df_preflop)} 种有样本的参数组合\n")
    display(df_preflop.head(20))

In [ ]:
def preflop_by_position(stats: PlayerStats) -> pd.DataFrame:
    """按位置汇总 Preflop 统计。"""
    all_params = PreFlopParams.get_all_params(stats.table_type)
    
    position_stats: dict[str, ActionStats] = {}
    
    for i, params in enumerate(all_params):
        # 只统计 previous_action == FOLD 的（即首次行动）
        if params.previous_action != ActionType.FOLD:
            continue
        
        pos_name = params.position.name
        if pos_name not in position_stats:
            position_stats[pos_name] = ActionStats()
        position_stats[pos_name].append(stats.preflop_stats[i])
    
    rows = []
    for pos_name, action_stats in position_stats.items():
        total = action_stats.total_samples()
        if total == 0:
            continue
        rows.append({
            "位置": pos_name,
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        })
    
    # 按位置顺序排序
    pos_order = {"SMALL_BLIND": 0, "BIG_BLIND": 1, "UTG": 2, "HJ": 3, "CO": 4, "BUTTON": 5}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_order"] = df["位置"].map(lambda x: pos_order.get(x, 99))
        df = df.sort_values("_order").drop(columns=["_order"])
    return df

if player_stats:
    print(f"\n📍 按位置汇总 Preflop 行动\n")
    df_by_pos = preflop_by_position(player_stats)
    display(df_by_pos)

## 4. Postflop 参数分析

In [ ]:
def analyze_postflop_stats(stats: PlayerStats) -> pd.DataFrame:
    """分析 Postflop 统计数据。"""
    all_params = PostFlopParams.get_all_params(stats.table_type)
    
    rows = []
    for i, params in enumerate(all_params):
        action_stats = stats.postflop_stats[i]
        total = action_stats.total_samples()
        if total == 0:
            continue
        
        row = {
            "索引": i,
            "街": params.street.name,
            "轮次": params.round,
            "前动作": params.prev_action.name,
            "下注数": params.num_bets,
            "IP": "是" if params.in_position else "否",
            "玩家数": params.num_players,
            "底池类型": params.preflop_pot_type.name,
            "PF攻击者": "是" if params.is_preflop_aggressor else "否",
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df.sort_values("样本数", ascending=False)

if player_stats:
    print(f"\n🎯 {player_stats.player_name} Postflop 参数分析\n")
    df_postflop = analyze_postflop_stats(player_stats)
    print(f"共 {len(df_postflop)} 种有样本的参数组合\n")
    display(df_postflop.head(20))

In [ ]:
def postflop_by_street(stats: PlayerStats) -> pd.DataFrame:
    """按街汇总 Postflop 统计。"""
    all_params = PostFlopParams.get_all_params(stats.table_type)
    
    street_stats: dict[str, ActionStats] = {}
    
    for i, params in enumerate(all_params):
        street_name = params.street.name
        if street_name not in street_stats:
            street_stats[street_name] = ActionStats()
        street_stats[street_name].append(stats.postflop_stats[i])
    
    rows = []
    for street_name, action_stats in street_stats.items():
        total = action_stats.total_samples()
        if total == 0:
            continue
        rows.append({
            "街": street_name,
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        })
    
    street_order = {"FLOP": 0, "TURN": 1, "RIVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_order"] = df["街"].map(lambda x: street_order.get(x, 99))
        df = df.sort_values("_order").drop(columns=["_order"])
    return df

if player_stats:
    print(f"\n🎴 按街汇总 Postflop 行动\n")
    df_by_street = postflop_by_street(player_stats)
    display(df_by_street)

In [ ]:
def postflop_by_position_and_street(stats: PlayerStats) -> pd.DataFrame:
    """按位置和街汇总 Postflop 统计。"""
    all_params = PostFlopParams.get_all_params(stats.table_type)
    
    # key: (in_position, street)
    grouped_stats: dict[tuple[bool, str], ActionStats] = {}
    
    for i, params in enumerate(all_params):
        key = (params.in_position, params.street.name)
        if key not in grouped_stats:
            grouped_stats[key] = ActionStats()
        grouped_stats[key].append(stats.postflop_stats[i])
    
    rows = []
    for (in_pos, street), action_stats in grouped_stats.items():
        total = action_stats.total_samples()
        if total == 0:
            continue
        rows.append({
            "位置": "IP" if in_pos else "OOP",
            "街": street,
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        })
    
    street_order = {"FLOP": 0, "TURN": 1, "RIVER": 2}
    pos_order = {"OOP": 0, "IP": 1}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_street_order"] = df["街"].map(lambda x: street_order.get(x, 99))
        df["_pos_order"] = df["位置"].map(lambda x: pos_order.get(x, 99))
        df = df.sort_values(["_street_order", "_pos_order"]).drop(columns=["_street_order", "_pos_order"])
    return df

if player_stats:
    print(f"\n📊 按位置和街汇总 Postflop 行动\n")
    df_pos_street = postflop_by_position_and_street(player_stats)
    display(df_pos_street)

## 5. 下注尺度分析

In [ ]:
def analyze_bet_sizing(stats: PlayerStats) -> pd.DataFrame:
    """分析下注尺度分布。"""
    all_postflop = PostFlopParams.get_all_params(stats.table_type)
    
    # 按街汇总下注尺度
    street_sizing: dict[str, dict[str, int]] = {}
    
    for i, params in enumerate(all_postflop):
        action_stats = stats.postflop_stats[i]
        street = params.street.name
        
        if street not in street_sizing:
            street_sizing[street] = {
                "0-40%": 0,
                "40-80%": 0,
                "80-120%": 0,
                ">120%": 0,
            }
        
        street_sizing[street]["0-40%"] += action_stats.bet_0_40
        street_sizing[street]["40-80%"] += action_stats.bet_40_80
        street_sizing[street]["80-120%"] += action_stats.bet_80_120
        street_sizing[street][">120%"] += action_stats.bet_over_120
    
    rows = []
    for street, sizing in street_sizing.items():
        total = sum(sizing.values())
        if total == 0:
            continue
        rows.append({
            "街": street,
            "总下注数": total,
            "0-40%": f"{sizing['0-40%']} ({sizing['0-40%']/total*100:.1f}%)",
            "40-80%": f"{sizing['40-80%']} ({sizing['40-80%']/total*100:.1f}%)",
            "80-120%": f"{sizing['80-120%']} ({sizing['80-120%']/total*100:.1f}%)",
            ">120%": f"{sizing['>120%']} ({sizing['>120%']/total*100:.1f}%)",
        })
    
    street_order = {"FLOP": 0, "TURN": 1, "RIVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_order"] = df["街"].map(lambda x: street_order.get(x, 99))
        df = df.sort_values("_order").drop(columns=["_order"])
    return df

if player_stats:
    print(f"\n💰 {player_stats.player_name} 下注尺度分析\n")
    df_sizing = analyze_bet_sizing(player_stats)
    display(df_sizing)

## 6. 底池类型分析

In [ ]:
def analyze_by_pot_type(stats: PlayerStats) -> pd.DataFrame:
    """按底池类型分析 Postflop 行为。"""
    all_params = PostFlopParams.get_all_params(stats.table_type)
    
    pot_type_stats: dict[str, ActionStats] = {}
    
    for i, params in enumerate(all_params):
        pot_type = params.preflop_pot_type.name
        if pot_type not in pot_type_stats:
            pot_type_stats[pot_type] = ActionStats()
        pot_type_stats[pot_type].append(stats.postflop_stats[i])
    
    rows = []
    for pot_type, action_stats in pot_type_stats.items():
        total = action_stats.total_samples()
        if total == 0:
            continue
        rows.append({
            "底池类型": pot_type,
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        })
    
    pot_order = {"LIMPED": 0, "SINGLE_RAISED": 1, "THREE_BET_PLUS": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_order"] = df["底池类型"].map(lambda x: pot_order.get(x, 99))
        df = df.sort_values("_order").drop(columns=["_order"])
    return df

if player_stats:
    print(f"\n🏆 按底池类型分析 Postflop 行为\n")
    df_pot_type = analyze_by_pot_type(player_stats)
    display(df_pot_type)

In [ ]:
def analyze_aggressor_vs_caller(stats: PlayerStats) -> pd.DataFrame:
    """分析作为 Preflop 攻击者 vs 跟注者的 Postflop 行为差异。"""
    all_params = PostFlopParams.get_all_params(stats.table_type)
    
    aggressor_stats = ActionStats()
    caller_stats = ActionStats()
    
    for i, params in enumerate(all_params):
        if params.is_preflop_aggressor:
            aggressor_stats.append(stats.postflop_stats[i])
        else:
            caller_stats.append(stats.postflop_stats[i])
    
    rows = []
    for role, action_stats in [("PF攻击者", aggressor_stats), ("PF跟注者", caller_stats)]:
        total = action_stats.total_samples()
        if total == 0:
            continue
        rows.append({
            "角色": role,
            "样本数": total,
            "加注%": f"{action_stats.bet_raise_probability() * 100:.1f}%",
            "跟注%": f"{action_stats.check_call_probability() * 100:.1f}%",
            "弃牌%": f"{action_stats.fold_probability() * 100:.1f}%",
        })
    
    return pd.DataFrame(rows)

if player_stats:
    print(f"\n⚔️ Preflop 攻击者 vs 跟注者 Postflop 行为对比\n")
    df_agg_vs_caller = analyze_aggressor_vs_caller(player_stats)
    display(df_agg_vs_caller)

## 7. 数据导出

In [ ]:
# 如需导出分析结果到 CSV，取消下面的注释

# if player_stats:
#     output_dir = project_root / "analysis" / "outputs"
#     output_dir.mkdir(parents=True, exist_ok=True)
#     
#     df_core.to_csv(output_dir / f"{PLAYER_NAME}_core_stats.csv", index=False)
#     df_preflop.to_csv(output_dir / f"{PLAYER_NAME}_preflop.csv", index=False)
#     df_postflop.to_csv(output_dir / f"{PLAYER_NAME}_postflop.csv", index=False)
#     
#     print(f"✅ 数据已导出到: {output_dir}")